# 03 Baselines: the floor every model must beat

Two trivial baselines (majority class and stratified dummy), evaluated with stratified 5-fold CV on the **training split only**. Validation and test splits are never loaded here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [ ]:
import pandas as pd  # noqa: E402
from sklearn.dummy import DummyClassifier  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (  # noqa: E402
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
)

from src.pipeline import SEED, build_pipeline, load_splits, split_xy  # noqa: E402

train, _, _ = load_splits()
X_train, y_train = split_xy(train)
print(X_train.shape, f"anomaly rate = {y_train.mean():.4f}")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}
baselines = {
    "Majority class": DummyClassifier(strategy="most_frequent"),
    "Stratified dummy": DummyClassifier(strategy="stratified", random_state=SEED),
}

In [ ]:
rows = {}
for name, model in baselines.items():
    res = cross_validate(
        build_pipeline(model), X_train, y_train, cv=cv, scoring=scoring
    )
    rows[name] = {m: res[f"test_{m}"] for m in scoring}

mean_table = pd.DataFrame(
    {n: {m: s.mean() for m, s in r.items()} for n, r in rows.items()}
).T
std_table = pd.DataFrame(
    {n: {m: s.std() for m, s in r.items()} for n, r in rows.items()}
).T
print("Mean over 5 folds")
print(mean_table.round(4))
print()
print("Std over 5 folds")
print(std_table.round(4))

In [ ]:
for name, model in baselines.items():
    pred = cross_val_predict(build_pipeline(model), X_train, y_train, cv=cv)
    print(name)
    print(confusion_matrix(y_train, pred))
    print()

## Takeaway

Accuracy is misleading here: always predicting "normal" scores about 96% accuracy but has recall 0 and F1 0. Both baselines have ROC-AUC near 0.50 and PR-AUC near 0.039 (the anomaly rate), so **0.039 is the PR-AUC floor** every real model must beat.